In [35]:
df.to_csv("generic_fraud_transactions.csv", index=False)


In [34]:
import pandas as pd
import numpy as np

np.random.seed(42)

N = 10_000  # dataset size

# Helper values
countries = ["US", "IN", "UK", "DE", "FR", "SG"]
merchant_categories = ["electronics", "fashion", "grocery", "travel", "gaming"]
device_types = ["mobile", "web", "pos"]
channels = ["online", "card_present", "in_app"]
payment_methods = ["credit_card", "debit_card", "upi", "wallet"]
currencies = ["USD", "INR", "EUR"]

df = pd.DataFrame({
    "transaction_id": np.arange(1, N + 1),
    "customer_id": np.random.randint(1000, 5000, N),
    "amount": np.round(np.random.exponential(scale=120, size=N), 2),
    "currency": np.random.choice(currencies, N),
    "transaction_time": pd.to_datetime("2024-01-01") + 
                        pd.to_timedelta(np.random.randint(0, 60*60*24*90, N), unit="s"),
    "merchant_id": np.random.randint(200, 800, N),
    "merchant_category": np.random.choice(merchant_categories, N),
    "merchant_country": np.random.choice(countries, N),
    "country": np.random.choice(countries, N),
    "device_type": np.random.choice(device_types, N),
    "channel": np.random.choice(channels, N),
    "payment_method": np.random.choice(payment_methods, N),
    "transaction_count_24h": np.random.poisson(lam=2, size=N),
    "avg_amount_24h": np.round(np.random.exponential(scale=100, size=N), 2),
})

# International flag
df["is_international"] = (df["country"] != df["merchant_country"]).astype(int)

# Fraud logic (RULE-BASED, realistic)
fraud_score = (
    (df["amount"] > 300).astype(int) +
    df["is_international"] +
    (df["device_type"] == "mobile").astype(int) +
    (df["channel"] == "online").astype(int) +
    (df["transaction_count_24h"] > 4).astype(int)
)

# Convert score to probability
fraud_probability = np.clip(fraud_score / 6, 0, 1)

# Final fraud label
df["is_fraud"] = (np.random.rand(N) < fraud_probability * 0.4).astype(int)

print("Fraud rate:", df["is_fraud"].mean())
df.head()


Fraud rate: 0.1086


,transaction_id,customer_id,amount,currency,transaction_time,merchant_id,merchant_category,merchant_country,country,device_type,channel,payment_method,transaction_count_24h,avg_amount_24h,is_international,is_fraud
0,1,4174,119.56,INR,2024-01-22 02:45:51,524,gaming,US,FR,web,card_present,upi,2,112.06,1,0
1,2,4507,72.98,USD,2024-03-11 18:55:26,566,gaming,UK,FR,mobile,in_app,upi,0,71.80,1,0
2,3,1860,78.94,USD,2024-03-27 10:32:50,535,grocery,SG,IN,pos,in_app,upi,6,6.85,1,0
3,4,2294,170.15,INR,2024-03-28 11:35:42,495,travel,SG,DE,mobile,in_app,debit_card,2,99.03,1,1
4,5,2130,39.74,INR,2024-02-13 04:21:10,223,grocery,IN,DE,mobile,card_present,debit_card,0,23.53,1,0


### Final Test Evaluation
The selected XGBoost model was evaluated on a held-out test set to estimate real-world fraud detection performance after all model decisions were finalized.


In [33]:
y_test_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

threshold = 0.9
y_test_pred_custom_xgb = (y_test_proba_xgb >= threshold).astype(int)

print(classification_report(y_test, y_test_pred_custom_xgb))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     42670
           1       0.93      0.73      0.82        52

    accuracy                           1.00     42722
   macro avg       0.96      0.87      0.91     42722
weighted avg       1.00      1.00      1.00     42722



In [32]:
from sklearn.metrics import classification_report

y_test_pred_xgb = xgb_model.predict(X_test)
print(classification_report(y_test, y_test_pred_xgb))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     42670
           1       0.87      0.75      0.80        52

    accuracy                           1.00     42722
   macro avg       0.93      0.87      0.90     42722
weighted avg       1.00      1.00      1.00     42722



### Model Comparison
XGBoost significantly improved fraud precision (~93%) compared to Logistic Regression while maintaining reasonable recall (~77%), demonstrating its ability to capture complex fraud patterns and reduce false positives.


In [31]:
from sklearn.metrics import classification_report

y_val_pred_xgb = xgb_model.predict(X_val)
print(classification_report(y_val, y_val_pred_xgb))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     42665
           1       0.93      0.77      0.84        56

    accuracy                           1.00     42721
   macro avg       0.97      0.88      0.92     42721
weighted avg       1.00      1.00      1.00     42721



In [30]:
xgb_model.fit(X_train, y_train)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [29]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42
)


In [28]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
scale_pos_weight


np.float64(518.1770833333334)

In [27]:
from xgboost import XGBClassifier


### Threshold Tuning Results
By increasing the fraud probability threshold, precision improved significantly (from ~7% to ~29%) while maintaining high recall (~88%), reducing false positives without sacrificing fraud detection.


In [26]:
from sklearn.metrics import classification_report

print(classification_report(y_val, y_val_pred_custom))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     42665
           1       0.29      0.88      0.44        56

    accuracy                           1.00     42721
   macro avg       0.65      0.94      0.72     42721
weighted avg       1.00      1.00      1.00     42721



In [25]:
import numpy as np

threshold = 0.9
y_val_pred_custom = (y_val_proba >= threshold).astype(int)


In [24]:
y_val_proba = baseline_model.predict_proba(X_val)[:, 1]


### Baseline Results
The baseline logistic regression achieved high fraud recall (~93%) but low precision, which is expected for an initial model prioritizing fraud detection over false positives.


In [23]:
from sklearn.metrics import classification_report

y_val_pred = baseline_model.predict(X_val)
print(classification_report(y_val, y_val_pred))


              precision    recall  f1-score   support

           0       1.00      0.98      0.99     42665
           1       0.07      0.93      0.13        56

    accuracy                           0.98     42721
   macro avg       0.53      0.96      0.56     42721
weighted avg       1.00      0.98      0.99     42721



In [22]:
baseline_model.fit(X_train, y_train)


C:\Users\fawaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
C:\Users\fawaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regres

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [21]:
baseline_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    n_jobs=-1
)


In [20]:
from sklearn.linear_model import LogisticRegression


### Split Validation
Due to time-based splitting, fraud rates differ slightly across train, validation, and test sets. 
This reflects real-world non-stationarity and is expected in fraud detection systems.


In [18]:
print("Train fraud %:", y_train.mean() * 100)
print("Val fraud %:", y_val.mean() * 100)
print("Test fraud %:", y_test.mean() * 100)


Train fraud %: 0.19261250777472363
Val fraud %: 0.1310830738980829
Test fraud %: 0.12171714807359206


In [17]:
X_train = train_df.drop("Class", axis=1)
y_train = train_df["Class"]

X_val = val_df.drop("Class", axis=1)
y_val = val_df["Class"]

X_test = test_df.drop("Class", axis=1)
y_test = test_df["Class"]


In [16]:
train_df = df.iloc[:train_end]
val_df   = df.iloc[train_end:val_end]
test_df  = df.iloc[val_end:]


In [15]:
n = len(df)
train_end = int(0.70 * n)
val_end   = int(0.85 * n)


In [14]:
df = df.sort_values("Time").reset_index(drop=True)


### Class Imbalance
The dataset is highly imbalanced, with only ~0.17% of transactions labeled as fraud. 
Therefore, accuracy is not an appropriate evaluation metric. 
We prioritize recall, precision, and PR-AUC, and apply class-weighted loss during training.


In [10]:
df['Class'].value_counts(normalize=True)


Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64

In [9]:
df['Class'].value_counts()


Class
0    284315
1       492
Name: count, dtype: int64

In [8]:
df.head()


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [7]:
df = pd.read_csv("../Data/creditcard.csv")


In [6]:
import os
os.getcwd()


'c:\\Credit Card Fraud Mlops\\Notebooks'